# Setup

In [1]:
# Tìm thư mục gốc của dự án để notebook biết nơi chứa mã nguồn và dữ liệu.
from pathlib import Path
import sys
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm
from PIL import Image

def find_project_root(max_up=6):
    p = Path.cwd()
    for _ in range(max_up + 1):
        if (p / "src").exists() and (p / "configs").exists():
            return p
        if p.parent == p:
            break
        p = p.parent
    return Path.cwd()

PROJECT_ROOT = find_project_root()

sys.path.append(str(PROJECT_ROOT))

from src.config import load_yaml
from src.io_utils import save_csv, save_numpy

pipeline_cfg = load_yaml(PROJECT_ROOT / "configs/pipeline_config.yaml")
reid_cfg = pipeline_cfg["reid"]

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)


Device: cpu


In [2]:
# Cell này thực hiện một bước cụ thể trong luồng pipeline, có thể là nạp dữ liệu, xử lý, hoặc lưu kết quả.
# !pip install git+https://github.com/KaiyangZhou/deep-person-reid.git


In [3]:
# Cell này thực hiện một bước cụ thể trong luồng pipeline, có thể là nạp dữ liệu, xử lý, hoặc lưu kết quả.
!pip install torchreid



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\DUNG\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [4]:
# Cell này thực hiện một bước cụ thể trong luồng pipeline, có thể là nạp dữ liệu, xử lý, hoặc lưu kết quả.
import sys
print(sys.executable)
import sys
!"{sys.executable}" -m pip install tensorboard
import sys
!"{sys.executable}" -m pip install tensorboard yacs gdown


C:\Users\DUNG\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\DUNG\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\DUNG\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [5]:
# Nạp mô hình ReID để trích đặc trưng từ ảnh nhân vật, sử dụng cho việc ghép người giữa camera.
import torchreid
print("torchreid imported successfully")


C:\Users\DUNG\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torchreid\reid\metrics\rank.py:11: UserWarning: Cython evaluation (very fast so highly recommended) is unavailable, now use python evaluation.
  warnings.warn(


torchreid imported successfully


# Load OSNet

In [6]:
# Nạp mô hình ReID để trích đặc trưng từ ảnh nhân vật, sử dụng cho việc ghép người giữa camera.
import torchreid

reid_model = torchreid.models.build_model(
    name=reid_cfg["model"],
    num_classes=1000,
    pretrained=True
)

reid_model = reid_model.to(device)
reid_model.eval()

print("Loaded Re-ID model:", reid_cfg["model"])


Successfully loaded imagenet pretrained weights from "C:\Users\DUNG/.cache\torch\checkpoints\osnet_x1_0_imagenet.pth"
Loaded Re-ID model: osnet_x1_0


# Transform ảnh

In [7]:
# Nạp mô hình ReID để trích đặc trưng từ ảnh nhân vật, sử dụng cho việc ghép người giữa camera.
from torchvision import transforms

input_h = reid_cfg["input_height"]
input_w = reid_cfg["input_width"]

transform = transforms.Compose([
    transforms.Resize((input_h, input_w)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


def extract_embedding(image_path):
    img = Image.open(image_path).convert("RGB")
    tensor = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        feat = reid_model(tensor)
        feat = torch.nn.functional.normalize(feat, p=2, dim=1)

    return feat.cpu().numpy()[0]


# Extract crop embeddings

In [8]:
# Cell này thực hiện một bước cụ thể trong luồng pipeline, có thể là nạp dữ liệu, xử lý, hoặc lưu kết quả.
rep_df = pd.read_csv(PROJECT_ROOT / "outputs/crops/representative_crops.csv")

embedding_rows = []
embeddings = []

for idx, row in tqdm(rep_df.iterrows(), total=len(rep_df)):
    emb = extract_embedding(row["crop_path"])
    embeddings.append(emb)

    embedding_rows.append({
        "embedding_index": idx,
        "camera_id": row["camera_id"],
        "local_track_id": int(row["local_track_id"]),
        "frame_id": int(row["frame_id"]),
        "timestamp_sec": float(row["timestamp_sec"]),
        "crop_path": row["crop_path"],
        "quality_score": float(row["quality_score"])
    })

embeddings = np.vstack(embeddings)
emb_meta_df = pd.DataFrame(embedding_rows)

save_numpy(embeddings, PROJECT_ROOT / "outputs/embeddings/crop_embeddings.npy")
save_csv(emb_meta_df, PROJECT_ROOT / "outputs/embeddings/crop_embeddings_meta.csv")

print("Embeddings shape:", embeddings.shape)
emb_meta_df.head()


  0%|          | 0/1820 [00:00<?, ?it/s]

  0%|          | 1/1820 [00:00<04:16,  7.09it/s]

  0%|          | 4/1820 [00:00<01:47, 16.84it/s]

  0%|          | 7/1820 [00:00<01:27, 20.80it/s]

  1%|          | 10/1820 [00:00<01:20, 22.43it/s]

...

100%|█████████▉| 1816/1820 [01:20<00:00, 24.60it/s]

100%|█████████▉| 1819/1820 [01:20<00:00, 24.53it/s]

100%|██████████| 1820/1820 [01:20<00:00, 22.61it/s]

Embeddings shape: (1820, 512)


,embedding_index,camera_id,local_track_id,frame_id,timestamp_sec,crop_path,quality_score
0,0,MOT17-02-FRCNN,1,44,1.433333,D:\hoiancomputervision\ticket_suspicion_projec...,0.993238
1,1,MOT17-02-FRCNN,1,43,1.400000,D:\hoiancomputervision\ticket_suspicion_projec...,0.989640
2,2,MOT17-02-FRCNN,1,6,0.166667,D:\hoiancomputervision\ticket_suspicion_projec...,0.983757
3,3,MOT17-02-FRCNN,1,5,0.133333,D:\hoiancomputervision\ticket_suspicion_projec...,0.976241
4,4,MOT17-02-FRCNN,1,7,0.200000,D:\hoiancomputervision\ticket_suspicion_projec...,0.969124


# Tracklet-level embedding

In [9]:
# Cell này thực hiện một bước cụ thể trong luồng pipeline, có thể là nạp dữ liệu, xử lý, hoặc lưu kết quả.
emb_meta_df = pd.read_csv(PROJECT_ROOT / "outputs/embeddings/crop_embeddings_meta.csv")
embeddings = np.load(PROJECT_ROOT / "outputs/embeddings/crop_embeddings.npy")

tracklet_emb_rows = []
tracklet_embs = []

for (camera_id, track_id), group in emb_meta_df.groupby(["camera_id", "local_track_id"]):
    idxs = group["embedding_index"].values
    embs = embeddings[idxs]

    mean_emb = embs.mean(axis=0)
    mean_emb = mean_emb / np.linalg.norm(mean_emb)

    emb_index = len(tracklet_embs)
    tracklet_embs.append(mean_emb)

    tracklet_emb_rows.append({
        "tracklet_embedding_index": emb_index,
        "camera_id": camera_id,
        "local_track_id": int(track_id),
        "num_crops": len(group),
        "mean_quality_score": float(group["quality_score"].mean())
    })

tracklet_embs = np.vstack(tracklet_embs)
tracklet_emb_df = pd.DataFrame(tracklet_emb_rows)

save_numpy(tracklet_embs, PROJECT_ROOT / "outputs/embeddings/tracklet_embeddings.npy")
save_csv(tracklet_emb_df, PROJECT_ROOT / "outputs/embeddings/tracklet_embeddings_meta.csv")

print("Tracklet embeddings:", tracklet_embs.shape)
tracklet_emb_df.head()


Tracklet embeddings: (364, 512)


,tracklet_embedding_index,camera_id,local_track_id,num_crops,mean_quality_score
0,0,MOT17-02-FRCNN,1,5,0.982400
1,1,MOT17-02-FRCNN,2,5,0.950629
2,2,MOT17-02-FRCNN,3,5,0.846067
3,3,MOT17-02-FRCNN,4,5,1.022948
4,4,MOT17-02-FRCNN,5,5,0.603006


# Some check

In [10]:
# Cell này thực hiện một bước cụ thể trong luồng pipeline, có thể là nạp dữ liệu, xử lý, hoặc lưu kết quả.
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

embedding_files = []
embedding_files += list((PROJECT_ROOT / "outputs").rglob("*embedding*.csv"))
embedding_files += list((PROJECT_ROOT / "outputs").rglob("*embedding*.npy"))
embedding_files += list((PROJECT_ROOT / "outputs").rglob("*embeddings*.csv"))
embedding_files += list((PROJECT_ROOT / "outputs").rglob("*embeddings*.npy"))

print("Embedding files found:")
for p in embedding_files:
    print(p)

for p in embedding_files:
    print("\nFILE:", p)
    if p.suffix == ".csv":
        df = pd.read_csv(p)
        print("Rows:", len(df))
        print("Columns:", list(df.columns)[:20])
        print(df.head())
    elif p.suffix == ".npy":
        arr = np.load(p, allow_pickle=True)
        print("Shape:", arr.shape)
        print("Dtype:", arr.dtype)


Embedding files found:
D:\hoiancomputervision\ticket_suspicion_project\outputs\embeddings\crop_embeddings_meta.csv
D:\hoiancomputervision\ticket_suspicion_project\outputs\embeddings\tracklet_embeddings_meta.csv
D:\hoiancomputervision\ticket_suspicion_project\outputs\embeddings\crop_embeddings.npy
D:\hoiancomputervision\ticket_suspicion_project\outputs\embeddings\tracklet_embeddings.npy
D:\hoiancomputervision\ticket_suspicion_project\outputs\embeddings\crop_embeddings_meta.csv
D:\hoiancomputervision\ticket_suspicion_project\outputs\embeddings\tracklet_embeddings_meta.csv
D:\hoiancomputervision\ticket_suspicion_project\outputs\embeddings\crop_embeddings.npy
D:\hoiancomputervision\ticket_suspicion_project\outputs\embeddings\tracklet_embeddings.npy

FILE: D:\hoiancomputervision\ticket_suspicion_project\outputs\embeddings\crop_embeddings_meta.csv
Rows: 1820
Columns: ['embedding_index', 'camera_id', 'local_track_id', 'frame_id', 'timestamp_sec', 'crop_path', 'quality_score']
   embedding_ind